# 🚀 The Honest Ring Benchmark: AdamV vs AdamW (Tartarus Arena)
Welcome to the official Kaggle benchmark for **AdamV** (Adam Vedic Optimizer).
This is the **Tartarus Arena**, an extreme stress test featuring a 100-layer MLP without residual connections.

### 1. JIT CUDA Compilation (The Fused Kernel)
Instead of running on pure Python and suffering from CPU-GPU sync overhead, we use PyTorch's `load_inline` to compile our Bakhshali Quartic Brake kernel directly on Kaggle's Nvidia GPUs on-the-fly!

In [ ]:
import torch
import torch.utils.cpp_extension
import time

print("🔥 Compilando o Fused CUDA Kernel do AdamV (JIT)... Isso levará cerca de 40 segundos na primeira vez.")
start_time = time.time()

cuda_source = """
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <cmath>
#include <ATen/cuda/CUDAContext.h>

#ifndef M_PI
#define M_PI 3.14159265358979323846
#endif

const int BLOCK_SIZE = 256;

template <typename scalar_t>
__global__ void adamv_prepare_kernel(
    scalar_t* __restrict__ exp_avg,
    scalar_t* __restrict__ exp_avg_sq,
    scalar_t* __restrict__ direcao_buffer,
    const scalar_t* __restrict__ grad,
    float beta1, float beta2, float bias_correction1, float bias_correction2, float eps, int numel) {
    
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < numel) {
        scalar_t g = grad[idx];
        scalar_t m = exp_avg[idx];
        scalar_t v = exp_avg_sq[idx];

        m = static_cast<scalar_t>(beta1) * m + static_cast<scalar_t>(1.0 - beta1) * g;
        v = static_cast<scalar_t>(beta2) * v + static_cast<scalar_t>(1.0 - beta2) * g * g;
        
        exp_avg[idx] = m;
        exp_avg_sq[idx] = v;

        scalar_t m_hat = m / static_cast<scalar_t>(bias_correction1);
        scalar_t v_hat = v / static_cast<scalar_t>(bias_correction2);
        
        direcao_buffer[idx] = m_hat / (sqrt(v_hat) + static_cast<scalar_t>(eps));
    }
}

template <typename scalar_t>
__global__ void adamv_update_kernel(
    scalar_t* __restrict__ params,
    const scalar_t* __restrict__ grad,
    const scalar_t* __restrict__ exp_avg_sq,
    const scalar_t* __restrict__ direcao_buffer,
    const scalar_t* __restrict__ norm_tensor_ptr,
    float progresso, float cooling_factor, float bakh_thresh_eff, float bias_correction2, float eps, 
    float wd_factor, float lr_max, float weight_decay, int numel, int D,
    bool omni_triggered, uint32_t punning_mask) {
    
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < numel) {
        scalar_t p = params[idx];
        
        // MACRO-BRANCH: Uniforme no warp, não gera divergence severa
        if (omni_triggered) {
            float p_f = static_cast<float>(p);
            // REGRA: Checagem de robustez antes de mexer na bitagem
            if (!isnan(p_f) && !isinf(p_f) && p_f != 0.0f) {
                uint32_t p_int = __float_as_uint(p_f);
                uint32_t sign = p_int & 0x80000000;
                uint32_t exp  = p_int & 0x7F800000;
                uint32_t mant = p_int & 0x007FFFFF;
                
                // Aplica a máscara APENAS na mantissa
                uint32_t mant_mod = mant & punning_mask;
                
                p_f = __uint_as_float(sign | exp | mant_mod);
                p = static_cast<scalar_t>(p_f);
            }
        }
        
        scalar_t g = grad[idx];
        scalar_t dir = direcao_buffer[idx];
        scalar_t v = exp_avg_sq[idx];
        
        scalar_t norm_dir = (*norm_tensor_ptr) / sqrt(static_cast<scalar_t>(D));
        scalar_t envelope = static_cast<scalar_t>(1.0) / (static_cast<scalar_t>(progresso) + norm_dir + static_cast<scalar_t>(eps));
        scalar_t lr_efetivo = static_cast<scalar_t>(lr_max) * min(envelope * static_cast<scalar_t>(cooling_factor), static_cast<scalar_t>(3.0));
        
        scalar_t a = lr_efetivo * dir;
        scalar_t v_hat = v / static_cast<scalar_t>(bias_correction2);
        scalar_t sqrt_v = sqrt(v_hat);
        
        bool explosao_mask = abs(g) > (static_cast<scalar_t>(bakh_thresh_eff) * sqrt_v);
        
        // Bakhshali Quartic Brake
        scalar_t denom = abs(p) + abs(a) + static_cast<scalar_t>(eps);
        scalar_t correction = (a * a) / (static_cast<scalar_t>(2.0) * denom);
        scalar_t bakhshali_brake = a - (a > static_cast<scalar_t>(0) ? static_cast<scalar_t>(1) : static_cast<scalar_t>(-1)) * correction;
        
        scalar_t step_size = explosao_mask ? bakhshali_brake : a;
        
        if (weight_decay != 0.0f) {
            p *= (static_cast<scalar_t>(1.0) - static_cast<scalar_t>(lr_max * weight_decay * wd_factor));
        }
        
        params[idx] = p - step_size;
    }
}

#define CHECK_CUDA(x) TORCH_CHECK(x.device().is_cuda(), #x " must be a CUDA tensor")
#define CHECK_CONTIGUOUS(x) TORCH_CHECK(x.is_contiguous(), #x " must be contiguous")
#define CHECK_INPUT(x) CHECK_CUDA(x); CHECK_CONTIGUOUS(x)

void adamv_step_cuda(
    at::Tensor p,
    at::Tensor grad,
    at::Tensor exp_avg,
    at::Tensor exp_avg_sq,
    at::Tensor direcao,
    float lr,
    float beta1,
    float beta2,
    float eps,
    float weight_decay,
    float progresso,
    float bakh_thresh_eff,
    int step,
    int D,
    bool omni_triggered,
    int64_t punning_mask) 
{
    CHECK_INPUT(p);
    CHECK_INPUT(grad);
    CHECK_INPUT(exp_avg);
    CHECK_INPUT(exp_avg_sq);
    CHECK_INPUT(direcao);

    int numel = p.numel();
    int blocks = (numel + BLOCK_SIZE - 1) / BLOCK_SIZE;

    float bias_correction1 = 1.0f - std::pow(beta1, step);
    float bias_correction2 = 1.0f - std::pow(beta2, step);

    cudaStream_t stream = at::cuda::getCurrentCUDAStream();

    AT_DISPATCH_FLOATING_TYPES_AND_HALF(p.scalar_type(), "adamv_prepare", [&] {
        adamv_prepare_kernel<scalar_t><<<blocks, BLOCK_SIZE, 0, stream>>>(
            exp_avg.data_ptr<scalar_t>(),
            exp_avg_sq.data_ptr<scalar_t>(),
            direcao.data_ptr<scalar_t>(),
            grad.data_ptr<scalar_t>(),
            beta1, beta2, bias_correction1, bias_correction2, eps, numel
        );
    });

    // Compute norm asynchronously on GPU
    at::Tensor norm_tensor = at::linalg_norm(direcao);
    
    float cooling_factor = 0.5f * (1.0f + std::cos(M_PI * progresso));
    float wd_factor = 0.5f * (1.0f + std::cos(M_PI * progresso));

    AT_DISPATCH_FLOATING_TYPES_AND_HALF(p.scalar_type(), "adamv_update", [&] {
        adamv_update_kernel<scalar_t><<<blocks, BLOCK_SIZE, 0, stream>>>(
            p.data_ptr<scalar_t>(),
            grad.data_ptr<scalar_t>(),
            exp_avg_sq.data_ptr<scalar_t>(),
            direcao.data_ptr<scalar_t>(),
            norm_tensor.data_ptr<scalar_t>(),
            progresso, cooling_factor, bakh_thresh_eff, bias_correction2, eps, wd_factor, lr, weight_decay, numel, D,
            omni_triggered, static_cast<uint32_t>(punning_mask)
        );
    });
}

"""

cpp_source = """
#include <torch/extension.h>
#include <ATen/cuda/CUDAContext.h>

void adamv_step_cuda(at::Tensor p, at::Tensor grad, at::Tensor exp_avg, at::Tensor exp_avg_sq, at::Tensor direcao, float lr, float beta1, float beta2, float eps, float weight_decay, float progresso, float bakh_thresh_eff, int step, int D, bool omni_triggered, int64_t punning_mask);
"""

adamv_cuda_module = torch.utils.cpp_extension.load_inline(
    name='adamv_cuda_jit',
    cpp_sources=cpp_source,
    cuda_sources=cuda_source,
    functions=['adamv_step_cuda'],
    with_cuda=True,
    extra_cflags=['-O3'],
    extra_cuda_cflags=['-O3', '-use_fast_math']
)
print(f"✅ CUDA Kernel compilado com sucesso em {time.time()-start_time:.2f} segundos!")



### 2. The AdamV Optimizer Source Code
We inject the Python wrapper of AdamV directly into this notebook. It will automatically detect the `adamv_cuda_module` compiled above and route all GPU tensors to it.

In [ ]:
import math
from torch.optim.optimizer import Optimizer


class AdamV(torch.optim.Optimizer):
    """
    AdamV (Adam Vedic) Optimizer - Pure Python Version.
    Combines Adam Momentum + Ramanujan Scale Envelope + Bakhshali Quartic Gate + OMNI Basin Hopping.
    """
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, 
                 weight_decay=0.01, total_steps=10000, 
                 bakhshali_threshold=3.0, enable_omni=True,
                 lp_kappa=0.1, lp_omega=10.0, punning_mask=0xFFFFE000):
                 
        if not 0.0 <= lr:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay,
                        total_steps=total_steps, bakhshali_threshold=bakhshali_threshold,
                        enable_omni=enable_omni, lp_kappa=lp_kappa, lp_omega=lp_omega, 
                        punning_mask=punning_mask)
        super(AdamV, self).__init__(params, defaults)
        
        self.state['omni_global'] = {
            'loss_ema': float('inf'),
            'patience': 0,
            'clock_reset_step': 0,
            'global_step': 0,
        }

    @torch.no_grad()
    def step(self, closure=None, current_loss=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
                
        if current_loss is not None:
            loss = current_loss

        g_state = self.state['omni_global']
        g_state['global_step'] += 1
        current_step = g_state['global_step']
        
        omni_triggered = False
        if loss is not None and len(self.param_groups) > 0 and self.param_groups[0]['enable_omni']:
            loss_val = loss.item() if isinstance(loss, torch.Tensor) else loss
            if g_state['loss_ema'] == float('inf'):
                g_state['loss_ema'] = loss_val
            else:
                g_state['loss_ema'] = 0.9 * g_state['loss_ema'] + 0.1 * loss_val
                
            if loss_val > g_state['loss_ema'] * 0.99:
                g_state['patience'] += 1
            else:
                g_state['patience'] = 0
                
            patience_limit = max(500, int(self.param_groups[0]['total_steps'] * 0.05))
            if g_state['patience'] >= patience_limit:
                omni_triggered = True
                g_state['patience'] = 0
                g_state['clock_reset_step'] = current_step
                
        for group in self.param_groups:
            lr_max = group['lr']
            beta1, beta2 = group['betas']
            eps = group['eps']
            weight_decay = group['weight_decay']
            total_steps = group['total_steps']
            bakh_thresh = group['bakhshali_threshold']
            lp_kappa = group['lp_kappa']
            lp_omega = group['lp_omega']
            punning_mask = group['punning_mask']
            
            internal_step = current_step - g_state['clock_reset_step']
            progresso = min(1.0, internal_step / max(1, total_steps))
            
            # Cálculo no HOST para evitar SFU Starvation na GPU
            LP_Fator = 1.0 + lp_kappa * math.cos(lp_omega * math.log(1.0 + progresso * 10.0))
            bakh_thresh_eff = bakh_thresh * LP_Fator
            
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    state['exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format)
                
                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                state['step'] += 1
                
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                
                bias_correction1 = 1 - beta1 ** state['step']
                bias_correction2 = 1 - beta2 ** state['step']
                
                m_hat = exp_avg / bias_correction1
                v_hat = exp_avg_sq / bias_correction2
                direcao = m_hat / (torch.sqrt(v_hat) + eps)
                
                if omni_triggered:
                    std_val = torch.std(p).item() if p.numel() > 1 else 0.05
                    noise = torch.randn_like(p) * (0.05 * std_val)
                    p.add_(noise)
                    # OMNI-ModBH puro no Python
                    p_int = p.view(torch.int32)
                    p_int.bitwise_and_(int(punning_mask))
                    pingala_scale = 2.0
                    state['exp_avg'].copy_(-noise * pingala_scale)
                    state['exp_avg_sq'].copy_(noise ** 2)
                
                D = p.numel()
                norm_dir_padrao = torch.linalg.norm(direcao) / math.sqrt(D)
                
                envelope = 1.0 / (progresso + norm_dir_padrao + eps)
                cooling_factor = 0.5 * (1.0 + math.cos(math.pi * progresso))
                lr_efetivo = lr_max * torch.clamp(envelope * cooling_factor, max=1.5)
                
                a = lr_efetivo * direcao
                
                sqrt_v = torch.sqrt(v_hat)
                explosao_mask = torch.abs(grad) > (bakh_thresh_eff * sqrt_v)
                
                denom = torch.abs(p) + torch.abs(a) + eps
                correction = (a ** 2) / (2.0 * denom)
                
                bakhshali_brake = a - torch.sign(a) * correction
                step_size = torch.where(explosao_mask, bakhshali_brake, a)
                
                if weight_decay != 0:
                    # Dynamic Endogenous Weight Decay (Cosine Annealing interno)
                    wd_factor = 0.5 * (1.0 + math.cos(math.pi * progresso))
                    p.data.mul_(1 - lr_max * weight_decay * wd_factor)
                    
                p.sub_(step_size)
                
        return loss

class AdamVCpp(torch.optim.Optimizer):
    """
    AdamV (Adam Vedic) Optimizer - C++ Fused Kernel Version.
    Extremely fast execution bypassing Python tensor loops.
    """
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, 
                 weight_decay=0.01, total_steps=10000, 
                 bakhshali_threshold=3.0, enable_omni=True,
                 lp_kappa=0.1, lp_omega=10.0, punning_mask=0xFFFFE000):
                 
        self.adamv_cpp = None
            
        self.adamv_cuda = adamv_cuda_module
            
        if not 0.0 <= lr:
            raise ValueError(f"Invalid learning rate: {lr}")
            
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay,
                        total_steps=total_steps, bakhshali_threshold=bakhshali_threshold,
                        enable_omni=enable_omni, lp_kappa=lp_kappa, lp_omega=lp_omega, 
                        punning_mask=punning_mask)
        super(AdamVCpp, self).__init__(params, defaults)
        
        self.state['omni_global'] = {
            'loss_ema': float('inf'),
            'patience': 0,
            'clock_reset_step': 0,
            'global_step': 0,
        }

    @torch.no_grad()
    def step(self, closure=None, current_loss=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        if current_loss is not None:
            loss = current_loss

        g_state = self.state['omni_global']
        g_state['global_step'] += 1
        current_step = g_state['global_step']
        
        omni_triggered = False
        if loss is not None and len(self.param_groups) > 0 and self.param_groups[0]['enable_omni']:
            loss_val = loss.item() if isinstance(loss, torch.Tensor) else loss
            if g_state['loss_ema'] == float('inf'):
                g_state['loss_ema'] = loss_val
            else:
                g_state['loss_ema'] = 0.9 * g_state['loss_ema'] + 0.1 * loss_val
                
            if loss_val > g_state['loss_ema'] * 0.99:
                g_state['patience'] += 1
            else:
                g_state['patience'] = 0
                
            patience_limit = max(500, int(self.param_groups[0]['total_steps'] * 0.05))
            if g_state['patience'] >= patience_limit:
                omni_triggered = True
                g_state['patience'] = 0
                g_state['clock_reset_step'] = current_step
                
        for group in self.param_groups:
            lr_max = group['lr']
            beta1, beta2 = group['betas']
            eps = group['eps']
            weight_decay = group['weight_decay']
            total_steps = group['total_steps']
            bakh_thresh = group['bakhshali_threshold']
            lp_kappa = group['lp_kappa']
            lp_omega = group['lp_omega']
            punning_mask = group['punning_mask']
            
            internal_step = current_step - g_state['clock_reset_step']
            progresso = min(1.0, internal_step / max(1, total_steps))
            
            LP_Fator = 1.0 + lp_kappa * math.cos(lp_omega * math.log(1.0 + progresso * 10.0))
            bakh_thresh_eff = bakh_thresh * LP_Fator
            
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    state['exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    state['direcao_buffer'] = torch.empty_like(p, memory_format=torch.preserve_format)
                
                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                state['step'] += 1
                
                if omni_triggered:
                    if p.numel() > 1:
                        std_val = torch.std(p)
                    else:
                        std_val = torch.tensor(0.05, dtype=p.dtype, device=p.device)
                        
                    noise = torch.randn_like(p) * (0.05 * std_val)
                    p.add_(noise)
                    # Note: OMNI-ModBH bitmask happens in CUDA kernel for GPU. 
                    pingala_scale = 2.0
                    state['exp_avg'].copy_(-noise * pingala_scale)
                    state['exp_avg_sq'].copy_(noise ** 2)
                
                if p.is_cpu:
                    self.adamv_cpp.adamv_step_cpu(
                        p, grad, exp_avg, exp_avg_sq, state['direcao_buffer'],
                        lr_max, beta1, beta2, eps, weight_decay,
                        float(progresso), float(bakh_thresh_eff), state['step'], p.numel()
                    )
                elif p.is_cuda and self.adamv_cuda is not None and hasattr(self.adamv_cuda, 'adamv_step_cuda'):
                    self.adamv_cuda.adamv_step_cuda(
                        p, grad, exp_avg, exp_avg_sq, state['direcao_buffer'],
                        lr_max, beta1, beta2, eps, weight_decay,
                        float(progresso), float(bakh_thresh_eff), state['step'], p.numel(),
                        bool(omni_triggered), int(punning_mask)
                    )
                else:
                    # Python fallback para GPU
                    if weight_decay != 0:
                        p.data.mul_(1 - lr_max * weight_decay)
                    
                    # Update moving averages FIRST
                    bias_correction1 = 1 - beta1 ** state['step']
                    bias_correction2 = 1 - beta2 ** state['step']
                    
                    exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                    exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                    
                    m_hat = exp_avg / bias_correction1
                    v_hat = exp_avg_sq / bias_correction2
                    direcao = m_hat / (v_hat.sqrt() + eps)
                    
                    norm_dir = torch.linalg.norm(direcao).item() / math.sqrt(p.numel())
                    envelope = 1.0 / (progresso + norm_dir + eps)
                    cooling = 0.5 * (1.0 + math.cos(math.pi * progresso))
                    lr_efetivo = lr_max * min(envelope * cooling, 3.0)
                    
                    a = lr_efetivo * direcao
                    sqrt_v = v_hat.sqrt()
                    
                    explosao_mask = torch.abs(grad) > (bakh_thresh_eff * sqrt_v)
                    
                    denom = torch.abs(p.data) + torch.abs(a) + eps
                    correction = (a * a) / (2.0 * denom)
                    bakhshali_brake = a - torch.sign(a) * correction
                    
                    step_size = torch.where(explosao_mask, bakhshali_brake, a)
                    
                    if weight_decay != 0:
                        wd_factor = 0.5 * (1.0 + math.cos(math.pi * progresso))
                        p.data.mul_(1.0 - lr_max * weight_decay * wd_factor)
                        
                    p.data.sub_(step_size)
                
        return loss



### 3. The Benchmark Setup (Tartarus Arena)
We will test AdamV against AdamW on a Tartarus MLP (100 layers, no residual connections) designed to suffer from catastrophic vanishing gradients.

In [ ]:
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

class TartarusMLP(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=512, num_layers=100, num_classes=10):
        super().__init__()
        layers = []
        for i in range(num_layers):
            in_f = input_dim if i == 0 else hidden_dim
            layers.append(nn.Linear(in_f, hidden_dim))
            layers.append(nn.GELU())
        layers.append(nn.Linear(hidden_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


In [ ]:
def run_tartarus_arena():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Running on {device}')
    
    N = 20000
    X = torch.randn(N, 512).to(device)
    # Create a real mathematical signal for the 10 classes
    signal = torch.sin(X[:, 0] * 5.0) + torch.cos(X[:, 1] * 5.0) + X[:, 2:10].sum(dim=1)
    # Use modulo to create perfectly balanced classes (0 to 9) with high non-linearity
    Y = (torch.abs(signal * 10).long()) % 10
    
    # Add 50% destructive noise
    noise_idx = torch.randperm(N)[:int(0.5 * N)]
    Y[noise_idx] = torch.randint(0, 10, (len(noise_idx),)).to(device)
    
    optimizers = ['AdamW', 'AdamV']
    results = {}
    
    import gc
    for opt_name in optimizers:
        torch.cuda.empty_cache()
        gc.collect()
        torch.manual_seed(42)
        model = TartarusMLP().to(device)
        
        if opt_name == 'AdamW':
            opt = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
        else:
            opt = AdamVCpp(model.parameters(), lr=1e-3, weight_decay=0.01)
            
        criterion = nn.CrossEntropyLoss()
        
        history = []
        start_time = time.time()
        
        for step in range(2000):
            # Mini-batch massivo para saturar a GPU (Reduzindo overhead de CPU)
            idx = torch.randint(0, 20000, (4096,), device=device)
            X_batch = X[idx]
            Y_batch = Y[idx]
            
            opt.zero_grad()
            out = model(X_batch)
            loss = criterion(out, Y_batch)
            loss.backward()
            
            if opt_name == 'AdamW':
                opt.step()
            else:
                opt.step(current_loss=loss.item())
                
            history.append(loss.item())
            
            if step % 200 == 0 or step == 1999:
                print(f'[{opt_name}] Step {step}/2000 | Loss: {loss.item():.4f}')
            
        results[opt_name] = history
        print(f'{opt_name} finished in {time.time()-start_time:.2f}s | Final Loss: {history[-1]:.4f}')
        
    plt.figure(figsize=(10,6))
    for name, hist in results.items():
        plt.plot(hist, label=name)
    plt.title('Tartarus Arena (100-Layer Deep MLP w/ 50% Noise)')
    plt.xlabel('Steps')
    plt.ylabel('CrossEntropy Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

run_tartarus_arena()
